In [ ]:
######### This code is used to analyze the importance values created by the ML models and create the base graphics #########

In [ ]:
## Import Packages
import pandas as pd
import numpy as np
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, Polygon

In [ ]:
##############################
###### Pull in the Base Data ######
##############################

In [ ]:
### Dams ### 
# Lists of IDs 
Selected_Dams = pd.read_csv(r"F:\Insert_File_Path\Final_Dam_List.csv", engine = 'python') # Update this file path
Final_Dams_List = Selected_Dams["DamID"].unique().tolist()

# Shapefile of Dams
GrodDams = gpd.read_file(r"F:\Insert_File_Path_of_Shapefile\Study_Dams.shp", columns=['grod_id', 'type', 'lon', 'lat'])  # Update this file path

# Filter Dams
AllDams = GrodDams[GrodDams['grod_id'].isin(Final_Dams_List)]

### River Centerlines ### 
SWORD = gpd.read_file(r"F:\Insert_File_Path_of_Shapefile\NA_SWORD_reach_v16_gt100.shp") # Update this file path
SWORD = SWORD.to_crs(4326)
bufferforplot = SWORD.buffer(0.1)

# Obstructed Nodes
Obst_Nodes = gpd.read_file(r"F:\Insert_File_Path_of_Shapefile\Obstructed_Nodes_Base_Clean.shp") # Update this file path
Obst_Nodes = Obst_Nodes.to_crs(4326)
Obst_Nodes_Sub = Obst_Nodes[Obst_Nodes['Assgn_dam'].isin(Final_Dams_List)]

### Mapping Basics ### 
USA = gpd.read_file(r"F:\Insert_File_Path_of_Shapefile\CONUS_Boundaries.shp") # Update this file path
# USA=USA.to_crs(4326)
CONUS = USA[(~USA['NAME'].isin([ 'Alaska','American Samoa', 'District of Columbia', 'Fed States of Micronesia', 'Guam', 'Hawaii','Marshall Islands', 'Northern Mariana Islands','Palau', 'Puerto Rico', 'Virgin Islands', 'District of Columbia']))]

In [ ]:
#### Plot Data -- Figure 1A ####
fig, ax=plt.subplots(figsize=(14,10))
CONUS.plot(ax=ax, color='whitesmoke', edgecolor='dimgray', linewidth = 0.7, alpha = 0.75, zorder = 1)
bufferforplot.plot(ax=ax,color='#006c7a',  linewidth = 0.01, alpha = 0.9,  label = "Rivers (>100m)", zorder = 3)
AllDams.plot(ax=ax, marker='^', markersize = 50, color = "#fb5607", edgecolor = "#5C3422",  linewidth = 0.5, label = "All Dams", zorder=4)
fig.legend(loc = "lower left")

fig.savefig(r"F:\Insert_File_Output_Path_Here\StudyArea.png", bbox_inches="tight", pad_inches=0.25, dpi=1200) # Update this file path

In [ ]:
###################################################################################################

In [ ]:
#### Get some information about the variables ####

In [ ]:
### Setup the Variable Dictionaries ###
var_categories = {'Dam.purpose':'Dam Operations', 'Dam.height': 'Dam Operations', 'Dam.storage':'Dam Operations' , 'Surface.area':'Dam Operations', 'Drainage.area': 'Dam Operations','Max.discharge': 'Dam Operations', 'Resv': 'Dam Operations',
          'Wind': 'Meterological', 'Soil.water.content': 'Meterological',  'Shortwave.radiation':'Meterological',  'Daily.maximum.air.temperature':'Meterological','Vapor.pressure': 'Meterological', 
          'Annual.mean.PET': 'Climate', 'Aridity': 'Climate' , 'High.precipitation.frequency':'Climate', 'High.precipitation.duration':'Climate', 'Seasonal.Precipitation':'Climate' , 'Seasonal.PET': 'Climate' , 'Seasonal.soil.water.content': 'Climate',
          'Up.Downstream.width.difference': 'Hydrology', 'Discharge': 'Hydrology','Annual.watershed.discharge':'Hydrology',  'Annual.watershed.runoff':'Hydrology', 'Inundation.extent':'Hydrology', 'Degree.of.regulation':'Hydrology', 'Groundwater.table.depth':'Hydrology',
          'Elevation': 'Landscape', 'Forest.extent':'Landscape' , 'Urban.extent': 'Landscape', 'Physiographic.Region': 'Landscape',
          'Month': 'Seasonality' , 'DOY_Sin': 'Seasonality', 'DOY_Cos': 'Seasonality'}

In [ ]:
##### Annual ######
## Note: DIR or  DIR UP represents "directly upstream" for comparisons

## Pull in Data ##
Dirup_Annual = pd.read_csv(r"F:\Insert_File_Path\Annual_Importance_Results.csv") # Update this file path

## Clean the Data ##
Dirup_Annual =  Dirup_Annual.rename(columns={'Unnamed: 0': 'Variable'})
Annual_Rank_DIR = Dirup_Annual.sort_values(['MDA' ], ascending=False ).reset_index(drop=True)
Annual_Rank_DIR["Rank"] = Annual_Rank_DIR["MDA"].rank(ascending=False).astype(int)
Annual_Rank_DIR["Annual"] = 'Annual'

## Get variable categories ##
Annual_Rank_DIR['Var_Cat'] = Annual_Rank_DIR['Variable'].map(var_categories)
Annual_Rank_DIR

In [ ]:
##### Seasonal ######

## Pull in the Data ##
Dirup_Winter = pd.read_csv(r"F:\Insert_File_Path\winter_importance_results.csv") # Update this file path
Dirup_Spring = pd.read_csv(r"F:\Insert_File_Path\spring_importance_results.csv") # Update this file path
Dirup_Summer = pd.read_csv(r"F:\Insert_File_Path\summer_importance_results.csv") # Update this file path
Dirup_Fall = pd.read_csv(r"F:\Insert_File_Path\fall_importance_results.csv") # Update this file path

In [ ]:
## Add Season to the DFs, Clean Columns, and Rank them ## 
Dirup_Winter['Season'] = 'Winter'
Dirup_Winter = Dirup_Winter.rename(columns={'Unnamed: 0': 'Variable'})
Winter_Rank = Dirup_Winter.sort_values(['MDA' ], ascending=False ).reset_index(drop=True)
Winter_Rank["Rank"] = Winter_Rank["MDA"].rank(ascending=False).astype(int)
Winter_Rank

Dirup_Spring['Season'] = 'Spring'
Dirup_Spring = Dirup_Spring.rename(columns={'Unnamed: 0': 'Variable'})
Spring_Rank = Dirup_Spring.sort_values(['MDA' ], ascending=False ).reset_index(drop=True)
Spring_Rank["Rank"] = Spring_Rank["MDA"].rank(ascending=False).astype(int)

Dirup_Summer['Season'] = 'Summer'
Dirup_Summer = Dirup_Summer.rename(columns={'Unnamed: 0': 'Variable'})
Summer_Rank = Dirup_Summer.sort_values(['MDA' ], ascending=False ).reset_index(drop=True)
Summer_Rank["Rank"] = Summer_Rank["MDA"].rank(ascending=False).astype(int)

Dirup_Fall['Season'] = 'Fall'
Dirup_Fall = Dirup_Fall.rename(columns={'Unnamed: 0': 'Variable'})
Fall_Rank = Dirup_Fall.sort_values(['MDA'], ascending=False ).reset_index(drop=True)
Fall_Rank["Rank"] = Fall_Rank["MDA"].rank(ascending=False).astype(int)


## Combine ##
Season_MDA_DIR = pd.concat([Winter_Rank,Spring_Rank,Summer_Rank,Fall_Rank])

## Get variable categories ##
Season_MDA_DIR['Var_Cat'] = Season_MDA_DIR['Variable'].map(var_categories)
Season_MDA_DIR

In [ ]:
### Save Tables ###
Annual_Rank_DIR.to_csv(r'F:\Insert_File_Output_Path_Here\Annual_Ranks.csv') # Update this file path
Season_MDA_DIR.to_csv(r'F:\Insert_File_Output_Path_Here\Seasonal_Ranks.csv') # Update this file path

In [ ]:
########################################################################################################################################

In [ ]:
####### Create the stacked plots #######

In [ ]:
### Annual  ---- Figure 2A ###
# Make a copy of the data 
Annual_MDA = Annual_Rank_DIR[:]

# Calculate total MDA
Total_Ann_MDA = Annual_MDA['MDA'].sum()
# Calculate  MDA% for each variable (Annual)
Annual_MDA['Per_MDA'] = 100 * (Annual_MDA['MDA']/Total_Ann_MDA)

## Get the MDA % for each category type
# Define the percentage function
def get_percentage(x):
    return 100 * (x.sum()/Total_Ann_MDA)

# Calculate
VarCat_MDAs = Annual_MDA.groupby(['Var_Cat']).agg({'MDA': get_percentage})
VarCat_MDAs.columns = ['Importance']
VarCat_MDAs = VarCat_MDAs.reset_index()
VarCat_MDAs['Annual'] = 'Annual'

# Preview
VarCat_MDAs

In [ ]:
print('Clim + Met: ' + str((VarCat_MDAs[VarCat_MDAs['Var_Cat'].isin(['Climate', "Meterological"])]).Importance.sum()))
print('All but Dam ops: ' + str((VarCat_MDAs[VarCat_MDAs['Var_Cat'] != 'Dam Operations' ]).Importance.sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(1, 6))

# Prep for Plotting 
Seas = VarCat_MDAs[VarCat_MDAs['Var_Cat'] == 'Seasonality'].Importance
Land = VarCat_MDAs[VarCat_MDAs['Var_Cat'] == 'Landscape'].Importance
Hydro = VarCat_MDAs[VarCat_MDAs['Var_Cat'] == 'Hydrology'].Importance
Clim = VarCat_MDAs[VarCat_MDAs['Var_Cat'] == 'Climate'].Importance
Met = VarCat_MDAs[VarCat_MDAs['Var_Cat'] == 'Meterological'].Importance
DOps = VarCat_MDAs[VarCat_MDAs['Var_Cat'] == 'Dam Operations'].Importance

# plot bars in stack manner
plt.bar('Annual', Seas, color = "#073b4c", edgecolor='whitesmoke')
plt.bar('Annual', Land, bottom = (Seas.iloc[0]), color="#118ab2", edgecolor='whitesmoke')
plt.bar('Annual', Hydro, bottom = (Seas.iloc[0] + Land.iloc[0]), color="#06d6a0", edgecolor='whitesmoke')
plt.bar('Annual', Clim, bottom = (Seas.iloc[0] + Land.iloc[0] + Hydro.iloc[0]), color="#ffd166",edgecolor='whitesmoke')
plt.bar('Annual', Met, bottom = (Seas.iloc[0] + Land.iloc[0] + Hydro.iloc[0] + Clim.iloc[0]), color="#f78c6b",edgecolor='whitesmoke')
plt.bar('Annual', DOps, bottom = (Seas.iloc[0] + Land.iloc[0] + Hydro.iloc[0] + Clim.iloc[0] + Met.iloc[0]), color="#ef476f",edgecolor='whitesmoke')
plt.ylabel("Relative Importance (%)")
plt.legend(['Seasonality', 'Landscape', 'Hydrology','Climate','Meterological', 'Dam Operations'], bbox_to_anchor=(1.05, 1) )
plt.show()

## Save the Plot ##
fig.savefig(r"F:\Insert_File_Output_Path_Here\Annual_Rel_Importance_Category.pdf", bbox_inches="tight", pad_inches=0.25) # Update this file path

In [ ]:
###############################################################

In [ ]:
### Seasonal   ---- Figure 2B ###
# Make a copy of the data 
Seasonal_MDA = Season_MDA_DIR[:]

VarCat_MDAs_Seas = pd.DataFrame()

# Calculate total MDA
Total_Seas_MDA = Seasonal_MDA.groupby(['Season']).agg({'MDA': 'sum'})
Total_Seas_MDA.columns = ['TotalMDA']
Total_Seas_MDA = Total_Seas_MDA.reset_index()

seasons = ['Winter', 'Spring', 'Summer', 'Fall']

for i in seasons: 
    Seas_Values = Seasonal_MDA[Seasonal_MDA['Season'] == i]
    Seas_Total = Total_Seas_MDA[Total_Seas_MDA['Season'] == i].TotalMDA.iloc[0]

    # Calculate  MDA% for each variable (Seasonal)
    Seas_Values['Per_MDA'] = 100 * (Seas_Values['MDA']/Seas_Total)

    ## Get the MDA % for each category type
    # Define the percentage function
    def get_percentage_seas(x):
        return 100 * (x.sum()/Seas_Total)

    # Calculate
    Seas_perc = Seas_Values.groupby(['Var_Cat']).agg({'MDA': get_percentage_seas})
    Seas_perc.columns = ['Importance']
    Seas_perc = Seas_perc.reset_index()
    Seas_perc['Season'] = i

    # Merge
    Output = pd.concat([VarCat_MDAs_Seas, Seas_perc], ignore_index=True)

    VarCat_MDAs_Seas = Output

# Preview
VarCat_MDAs_Seas

In [ ]:
print("Winter Clim + Met: " + str(VarCat_MDAs_Seas[(VarCat_MDAs_Seas['Season'] == 'Winter') & (VarCat_MDAs_Seas['Var_Cat'].isin(["Climate", 'Meterological']))].Importance.sum()))
print("Spring Clim + Met: " + str(VarCat_MDAs_Seas[(VarCat_MDAs_Seas['Season'] == 'Spring') & (VarCat_MDAs_Seas['Var_Cat'].isin(["Climate", 'Meterological']))].Importance.sum()))
print("Summer Clim + Met: " + str(VarCat_MDAs_Seas[(VarCat_MDAs_Seas['Season'] == 'Summer') & (VarCat_MDAs_Seas['Var_Cat'].isin(["Climate", 'Meterological']))].Importance.sum()))
print("Fall Clim + Met: " + str(VarCat_MDAs_Seas[(VarCat_MDAs_Seas['Season'] == 'Fall') & (VarCat_MDAs_Seas['Var_Cat'].isin(["Climate", 'Meterological']))].Importance.sum()))

In [ ]:
print("Winter All but Dam Ops: " + str(VarCat_MDAs_Seas[(VarCat_MDAs_Seas['Season'] == 'Winter') & (VarCat_MDAs_Seas['Var_Cat'] != 'Dam Operations')].Importance.sum()))
print("Spring All but Dam Ops: " + str(VarCat_MDAs_Seas[(VarCat_MDAs_Seas['Season'] == 'Spring') & (VarCat_MDAs_Seas['Var_Cat'] != 'Dam Operations')].Importance.sum()))
print("Summer All but Dam Ops: " + str(VarCat_MDAs_Seas[(VarCat_MDAs_Seas['Season'] == 'Summer') & (VarCat_MDAs_Seas['Var_Cat'] != 'Dam Operations')].Importance.sum()))
print("Fal lAll but Dam Ops: " + str(VarCat_MDAs_Seas[(VarCat_MDAs_Seas['Season'] == 'Fall') & (VarCat_MDAs_Seas['Var_Cat'] != 'Dam Operations')].Importance.sum()))

In [ ]:
# Set up plot specs
fig, ax = plt.subplots(figsize=(8, 6))

# Get arrays for plotting
Seas_Seas = VarCat_MDAs_Seas[VarCat_MDAs_Seas['Var_Cat'] == 'Seasonality'].loc[:, 'Importance'].values
Land_Seas = VarCat_MDAs_Seas[VarCat_MDAs_Seas['Var_Cat'] == 'Landscape'].loc[:, 'Importance'].values
Hydro_Seas = VarCat_MDAs_Seas[VarCat_MDAs_Seas['Var_Cat'] == 'Hydrology'].loc[:, 'Importance'].values
Clim_Seas = VarCat_MDAs_Seas[VarCat_MDAs_Seas['Var_Cat'] == 'Climate'].loc[:, 'Importance'].values
Met_Seas = VarCat_MDAs_Seas[VarCat_MDAs_Seas['Var_Cat'] == 'Meterological'].loc[:, 'Importance'].values
DOps_Seas = VarCat_MDAs_Seas[VarCat_MDAs_Seas['Var_Cat'] == 'Dam Operations'].loc[:, 'Importance'].values

# Get colors and labels
Category_Colors =[ "#073b4c","#118ab2","#06d6a0","#ffd166","#f78c6b", "#ef476f"]
Category_Labels = ['Seasonality', 'Landscape', 'Hydrology','Climate', 'Meterological', 'Dam Operations']

# Create the stack plot
plt.stackplot(seasons, Seas_Seas, Land_Seas, Hydro_Seas, Clim_Seas, Met_Seas, DOps_Seas,
               labels = Category_Labels, colors = Category_Colors, edgecolor = 'whitesmoke')

# Add labels and title
plt.xlabel('Season')
plt.ylabel('Relative Importance (%)')
plt.legend(bbox_to_anchor=(1.05, 1) )

# Show the plot
plt.show()

## Save the Plot ##
fig.savefig(r"F:\Insert_File_Output_Path_Here\Seasonal_Rel_Importance_Category.pdf", bbox_inches="tight", pad_inches=0.25) # Update this file path

In [ ]:
###############################################################

In [ ]:
###  Annual --- Figure 3A ###
## Get just the Dam Operations Variables and MDA Percentages ##
Annual_MDA_DOps = Annual_MDA[Annual_MDA['Var_Cat'] == 'Dam Operations']
Annual_MDA_DOps_pivot = Annual_MDA_DOps.pivot(index='Annual', columns='Variable', values='Per_MDA')
Annual_MDA_DOps_pivot = Annual_MDA_DOps_pivot[[ 'Dam.purpose', 'Dam.height', 'Dam.storage',  'Surface.area',  'Drainage.area', 'Max.discharge',  'Resv' ]]
Annual_MDA_DOps_pivot

In [ ]:
### Plot The Data ###
## Set up color dictionary
DamOp_Color_Dict = {'Dam.purpose':'#420d1a', 'Dam.height':'#ef476f', 'Dam.storage':"#91567e", 'Surface.area': '#a53860', 
                    'Drainage.area': "#662147", 'Max.discharge': '#ff8fab', 'Resv': '#d03d60'}

# Set up Figure
ax = Annual_MDA_DOps_pivot.plot(kind="bar", stacked=True, color=[DamOp_Color_Dict[col] for col in Annual_MDA_DOps_pivot.columns],  
                                edgecolor="white", linewidth=0.25, figsize=(1.5, 6))
ax.set_ylim(0,35)
ax.set_xlabel("")
ax.set_ylabel("Relative Importance (%)")
ax.legend(title="Variables", bbox_to_anchor=(1.05, 1), loc="upper left")

# Save Figure
fig = ax.get_figure()
fig.savefig(r"F:\Insert_File_Output_Path_Here\Annual_Dam_Ops_Rel_Imp.pdf", bbox_inches="tight", pad_inches=0.25) # Update this file path

In [ ]:
### Seasonal ---  Figure 3B ###
## Get just the Dam Operations Variables and MDA Percentages ##
Total_Seas_MDA = Seasonal_MDA.groupby(['Season']).agg({'MDA': 'sum'})
Total_Seas_MDA.columns = ['TotalMDA']
Total_Seas_MDA = Total_Seas_MDA.reset_index()

Seasonal_PerMDA = pd.DataFrame()

for i in seasons: 
    Seas_Values = Seasonal_MDA[Seasonal_MDA['Season'] == i]
    Seas_Total = Total_Seas_MDA[Total_Seas_MDA['Season'] == i].TotalMDA.iloc[0]

    # Calculate  MDA% for each variable (Seasonal)
    Seas_Values['Per_MDA'] = 100 * (Seas_Values['MDA']/Seas_Total)

    # Merge
    Output = pd.concat([Seasonal_PerMDA, Seas_Values], ignore_index=True)

    Seasonal_PerMDA = Output

## Get Just the Dam Operations 
Seasonal_MDA_DOps = Seasonal_PerMDA[Seasonal_PerMDA['Var_Cat'] == 'Dam Operations']

# Format the data for the plots
Seasonal_MDA_DOps_pivot = Seasonal_MDA_DOps.pivot(index='Season', columns='Variable', values='Per_MDA')
Seasonal_MDA_DOps_pivot = Seasonal_MDA_DOps_pivot[[ 'Dam.purpose', 'Dam.height', 'Dam.storage',  'Surface.area',  'Drainage.area', 'Max.discharge',  'Resv' ]]
Seasonal_MDA_DOps_pivot = Seasonal_MDA_DOps_pivot.reset_index()

order = ["Winter", "Spring", "Summer", "Fall"]
Seasonal_MDA_DOps_pivot["Season"] = pd.Categorical(Seasonal_MDA_DOps_pivot["Season"], categories=order, ordered=True)
Seasonal_MDA_DOps_pivot_ordered = Seasonal_MDA_DOps_pivot.sort_values("Season")
Seasonal_MDA_DOps_pivot_ordered = Seasonal_MDA_DOps_pivot_ordered.reset_index()
Seasonal_MDA_DOps_pivot_ordered = Seasonal_MDA_DOps_pivot_ordered[[ 'Dam.purpose', 'Dam.height', 'Dam.storage',  'Surface.area',  'Drainage.area', 'Max.discharge',  'Resv']]
Seasonal_MDA_DOps_pivot_ordered

In [ ]:
### Plot The Data ###
## Set up color dictionary
DamOp_Color_Dict = {'Dam.purpose':'#420d1a', 'Dam.height':'#ef476f', 'Dam.storage':"#91567e", 'Surface.area': '#a53860', 
                    'Drainage.area': "#662147", 'Max.discharge': '#ff8fab', 'Resv': '#d03d60'}

# Set up Figure
ax = Seasonal_MDA_DOps_pivot_ordered.plot(kind="bar", stacked=True, color=[DamOp_Color_Dict[col] for col in Seasonal_MDA_DOps_pivot_ordered.columns],  
                                edgecolor="white", linewidth= 0.25, figsize=(8, 6))

ax.set_ylabel("Relative Importance (%)")
ax.legend(title="Variables", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.set_xticks(range(len(seasons))) # Set numerical locations for ticks
ax.set_xticklabels(seasons) # Apply the custom label
ax.set_ylim(0,35)

# Save Figure
fig = ax.get_figure()
fig.savefig(r"F:\Insert_File_Output_Path_Here\Seasonal_Dam_Ops_Rel_Imp.pdf", bbox_inches="tight", pad_inches=0.25) # Update this file path

In [ ]:
###################################################################################

In [ ]:
#### Get some supplemental details ####

In [ ]:
## Bring in the DS Changes ## 
Clean_ML_Inputs = pd.read_csv(r"F:\Insert_File_Path\Temp_Change_Attributes_All.csv") # Update this file path
Clean_ML_Inputs_Signif_Sub = Clean_ML_Inputs[Clean_ML_Inputs['Sig_05'] == 'Significant']
Clean_ML_Inputs_Signif_Sub = Clean_ML_Inputs_Signif_Sub.dropna()

# Get Diff Direction
DS_Diff = Clean_ML_Inputs_Signif_Sub[['Date', 'Month', 'Assgn_dam', 'Temp_Diff']]

# Get Season
DS_Diff["Season"] = np.nan
DS_Diff["Season"] = np.where(DS_Diff["Month"].isin([12, 1, 2]), "Winter", DS_Diff["Season"])
DS_Diff["Season"] = np.where(DS_Diff["Month"].isin([3, 4, 5]), "Spring", DS_Diff["Season"])
DS_Diff["Season"] = np.where(DS_Diff["Month"].isin([6,7,8]), "Summer", DS_Diff["Season"])
DS_Diff["Season"] = np.where(DS_Diff["Month"].isin([9, 10, 11]), "Fall", DS_Diff["Season"])

# Preview
DS_Diff

In [ ]:
## Print Details 
print("Number of Winter: ", str(len(DS_Diff[DS_Diff['Season'] == 'Winter'])))
print("Number of Spring: ", str(len(DS_Diff[DS_Diff['Season'] == 'Spring'])))
print("Number of Summer: ", str(len(DS_Diff[DS_Diff['Season'] == 'Summer'])))
print("Number of Fall: ", str(len(DS_Diff[DS_Diff['Season'] == 'Fall'])))

In [ ]:
### Get the Descriptive Stats ###
# create a range function
def calculate_range(x):
    return x.max() - x.min()

## Get Group Info
Profile_Stats_Season = DS_Diff.groupby(["Season"]).agg({'Temp_Diff':['count','median','mean', 'min', 'max', 'std', calculate_range]})
Profile_Stats_Season.columns = ['Profile_Count','Median_Diff','Avg_Diff', 'Min', 'Max', 'STDV', 'Range']
Profile_Stats_Season = Profile_Stats_Season.reset_index()
Profile_Stats_Season

In [ ]:
## Get Annual Info ##
Profile_Stats_Ann = DS_Diff.agg({'Temp_Diff':['count','median','mean', 'min', 'max', 'std', calculate_range]})
Profile_Stats_Ann

In [ ]:
### Plot the spread of the differences by direction ###

# Set the plot details 
ColorPalette = ["#275da2", "#ea93b9", "#1ca390", "#dcaf68"]
HueOrder = ["Winter","Spring", "Summer", "Fall"]

# Plot
fig, ax=plt.subplots(figsize=(3,5))
ax = sns.boxenplot(data=DS_Diff, x='Season', y='Temp_Diff', hue = "Season", hue_order = HueOrder, palette = ColorPalette, linecolor= "#1F201F", width = 0.75, flier_kws=dict(facecolor="#1F201F", linewidth=.2))
ax.set_xlabel( "") 
ax.set_ylabel( "Downstream Temperature Difference (°C)", fontsize=12) 
ax.tick_params(labelsize=10)
ax.legend(title=None, loc = 'lower right' ,fontsize=10)

# Save Figure
fig.savefig(r"F:\Insert_File_Output_Path_Here\Seasonal_Temp_Diffs.pdf", bbox_inches="tight", pad_inches=0.25) # Update this file path

In [ ]:
#######################################

In [ ]:
## Dam Type Info ##
# Observation Info
info =pd.read_csv( r"F:\Insert_File_Path\Temp_Change_Attributes_All.csv") # Update this file path
info_sig = info[info['Sig_05'] == 'Significant']
info_sig = info_sig.dropna()

# Get profile counts
dam_type_count = info_sig.groupby(['Dam purpose']).agg({'Assgn_dam': 'count'})
dam_type_count.columns = ['Dam Type Count']
dam_type_count = dam_type_count.reset_index()
dam_type_count

In [ ]:
# Get dam counts
dam_type_prof_count = info_sig.groupby(['Dam purpose','Assgn_dam']).agg({'Assgn_dam': 'count'})
dam_type_prof_count.columns = ['Prof Count']
dam_type_prof_count = dam_type_prof_count.reset_index()

unique_dam_type = dam_type_prof_count.groupby(['Dam purpose']).agg({'Assgn_dam': 'count'})
unique_dam_type.columns = ['Dam Count']
unique_dam_type = unique_dam_type.reset_index()
unique_dam_type